In [ ]:
# Cell 1: Imports, settings, seed handling, and HemoBoundary reconstruction
from __future__ import annotations

import gc

import hashlib

import json

import math

import os

import warnings

from dataclasses import dataclass

from pathlib import Path

from types import SimpleNamespace

from typing import Any, Dict, List, Mapping, Sequence, Tuple

import numpy as np

import pandas as pd

from scipy import signal

from sklearn.linear_model import LogisticRegression, Ridge

SEEDS = tuple(range(1, 11))

SEED_OUTPUT_DIRNAME = 'all_seeds_01_10'

DRIVE_ROOT = Path('/content/gdrive/MyDrive')

SHIN_ROOT = DRIVE_ROOT / 'HonorProject' / 'HemoBoundary' / 'SHIN'

OUTPUT_ROOT = SHIN_ROOT / 'analysis/Functional_Necessity'

OUT_DIR = OUTPUT_ROOT / SEED_OUTPUT_DIRNAME

ALL_PIDS = tuple((f'VP{i:03d}' for i in range(1, 27)))

GRID_DT = 0.5

GRID_END = 20.0

GRID = np.arange(0.0, GRID_END + GRID_DT / 2.0, GRID_DT)

MIN_GAP = 0.5

MISS_PENALTY_SEC = 4.0

CHANGE_WINDOWS = (3, 5, 9)

PREDICTOR_KEYS = ('obs', 'state_feat')

LABEL_KEYS = ('present', 'onset', 'peak', 'offset', 'censored', 'states')

BASE_MODEL_RANDOM_SEED = 20260912

ACTIVE_RUN_SEED = 1

LOCAL_OOF_FOLDS_COHERENT = 3

ROW_FULL = 'BASE_PLUS_A_PLUS_B_PLUS_C_PLUS_LBR'

def mount_drive():
    if DRIVE_ROOT.is_dir():
        return
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=False)

def atomic_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{os.getpid()}')
    compression = 'gzip' if str(path).endswith('.gz') else None
    df.to_csv(tmp, index=False, compression=compression)
    os.replace(tmp, path)

def atomic_json(obj: Mapping[str, Any], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{os.getpid()}')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    os.replace(tmp, path)

def base_stable_seed(*parts: Any) -> int:
    raw = '|'.join((str(x) for x in parts)).encode('utf-8')
    return int(hashlib.sha256(raw).hexdigest()[:8], 16) % (2 ** 31 - 1)

def stable_seed(*parts: Any) -> int:
    if int(ACTIVE_RUN_SEED) == 1:
        return base_stable_seed(*parts)
    return base_stable_seed('MULTISEED_REPEAT', int(ACTIVE_RUN_SEED), *parts)

def configure_seed(seed: int):
    global ACTIVE_RUN_SEED
    ACTIVE_RUN_SEED = int(seed)
    np.random.seed(base_stable_seed('numpy_legacy', int(seed)) % (2 ** 32 - 1))
    hb.RANDOM_SEED = int(BASE_MODEL_RANDOM_SEED) + int(seed) - 1

def canonical_cfg(cfg: Mapping[str, Any]) -> str:
    return json.dumps({k: cfg[k] for k in sorted(cfg)}, sort_keys=True, separators=(',', ':'))

def finite_model_array(x: np.ndarray) -> np.ndarray:
    a = np.asarray(x)
    if not np.issubdtype(a.dtype, np.number):
        return a
    return np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0, copy=True)

def predictor_view(data: Mapping[str, np.ndarray]) -> Dict[str, np.ndarray]:
    return {k: data[k] for k in PREDICTOR_KEYS}

def require_predictors_only(x: Mapping[str, np.ndarray]) -> None:
    if set(x) != set(PREDICTOR_KEYS):
        raise RuntimeError(f'Prediction payload must contain ONLY {PREDICTOR_KEYS}; got {sorted(x)}')

def grouped_folds(pids: Sequence[str], seed: int, n: int=3) -> List[List[str]]:
    p = np.array(sorted(set(map(str, pids))), dtype=object)
    if len(p) < 3:
        raise RuntimeError('At least three source participants are required.')
    np.random.default_rng(seed).shuffle(p)
    return [q.tolist() for q in np.array_split(p, min(n, len(p))) if len(q)]

def subset(data: Mapping[str, np.ndarray], mask: np.ndarray) -> Dict[str, np.ndarray]:
    return {k: np.asarray(v)[mask] for k, v in data.items()}

def cat(parts: Sequence[Mapping[str, np.ndarray]]) -> Dict[str, np.ndarray]:
    return {k: np.concatenate([p[k] for p in parts], axis=0) for k in parts[0]}

def training_rows(caches: Mapping[str, Any], pids: Sequence[str]) -> Dict[str, np.ndarray]:
    return cat([caches[p]['train'] for p in sorted(pids)])

class FitAudit:

    def __init__(self, allowed: Sequence[str], forbidden: Sequence[str], context: str):
        self.allowed = set(allowed)
        self.forbidden = set(forbidden)
        self.context = context
        self.rows = []

    def record(self, data: Mapping[str, np.ndarray], stage: str, held: Sequence[str]=()):
        actual = set(map(str, data['pid']))
        blocked = self.forbidden | set(held)
        if not actual or not actual.issubset(self.allowed) or actual & blocked:
            raise RuntimeError(f'{self.context}/{stage}: invalid fit membership {actual & blocked}')
        self.rows.append(dict(context=self.context, stage=stage, fit_participants='|'.join(sorted(actual)), forbidden='|'.join(sorted(blocked)), n_rows=int(len(data['pid']))))

def smooth_series(x: np.ndarray, window: int) -> np.ndarray:
    x = np.asarray(x, float)
    if int(window) <= 1:
        return x
    w = min(int(window), len(x) if len(x) % 2 == 1 else len(x) - 1)
    if w < 3:
        return x
    if w % 2 == 0:
        w -= 1
    return signal.savgol_filter(x, w, 2 if w >= 5 else 1, mode='interp')

def change_branch_features(data: Mapping[str, np.ndarray], frac: float) -> np.ndarray:
    F = finite_model_array(data['state_feat']).astype(float, copy=False)
    anti = F[:, :, 2]
    rows: List[List[float]] = []
    for a in anti:
        scale_rows = []
        for w in CHANGE_WINDOWS:
            z = smooth_series(a, int(w))
            dz = np.gradient(z, GRID_DT)
            ddz = np.gradient(dz, GRID_DT)
            early = np.flatnonzero(GRID <= 8.0)
            oi = int(early[np.argmax(dz[early])]) if len(early) else int(np.argmax(dz))
            pi = int(oi + np.argmax(z[oi:]))
            mx = float(z[pi])
            aft = np.arange(pi + 1, len(z))
            offi = len(z) - 1
            if len(aft) and mx > 0:
                cand = aft[z[aft] <= float(frac) * mx]
                if len(cand):
                    offi = int(cand[0])
            feat = [float(GRID[oi]), float(GRID[pi]), float(GRID[offi]), float(np.max(dz)), float(np.min(dz)), mx, float(np.max(ddz)), float(np.min(ddz)), float(np.mean(np.abs(dz)))]
            scale_rows.append(feat)
        A = np.asarray(scale_rows, float)
        flat = A.reshape(-1).tolist()
        for col in (0, 1, 2):
            xx = A[:, col]
            flat.extend([float(np.median(xx)), float(np.quantile(xx, 0.75) - np.quantile(xx, 0.25)), float(np.max(xx) - np.min(xx))])
        flat.extend([float(np.mean(A[:, 3])), float(np.mean(A[:, 4])), float(np.mean(A[:, 5])), float(np.std(A[:, 3])), float(np.std(A[:, 4]))])
        rows.append(flat)
    return finite_model_array(np.asarray(rows, float))

def enforce_geometry(on: np.ndarray, pk: np.ndarray, off: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    on = np.asarray(on, float).copy()
    pk = np.asarray(pk, float).copy()
    off = np.asarray(off, float).copy()
    on = np.clip(on, 0.0, GRID_END - 2 * MIN_GAP)
    pk = np.maximum(pk, on + MIN_GAP)
    pk = np.clip(pk, MIN_GAP, GRID_END - MIN_GAP)
    off = np.maximum(off, pk + MIN_GAP)
    off = np.clip(off, 2 * MIN_GAP, GRID_END)
    return (on, pk, off)

class StableScaler:

    def fit(self, X, sample_weight=None):
        X = np.asarray(X, float)
        if X.ndim != 2 or not np.isfinite(X).all():
            raise RuntimeError('Scaler needs finite 2D training features.')
        w = np.ones(len(X), float) if sample_weight is None else np.asarray(sample_weight, float)
        if len(w) != len(X) or not np.isfinite(w).all() or (w < 0).any() or (w.sum() <= 0):
            raise RuntimeError('Invalid source sample weights.')
        self.mean_ = np.average(X, axis=0, weights=w)
        self.var_ = np.average((X - self.mean_) ** 2, axis=0, weights=w)
        self.var_ = np.maximum(self.var_, 0.0)
        eps = np.finfo(float).eps
        nn = float(w.sum())
        constant = self.var_ <= nn * eps * self.var_ + (nn * self.mean_ * eps) ** 2
        self.scale_ = np.sqrt(self.var_)
        self.scale_[constant | (self.scale_ == 0)] = 1.0
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = np.asarray(X, float)
        if X.ndim != 2 or X.shape[1] != self.n_features_in_ or (not np.isfinite(X).all()):
            raise RuntimeError('Scaler feature schema/finite check failed.')
        return (X - self.mean_) / self.scale_

def fixed_order_path(p, stay_bias=0.0):
    logp = np.log(np.clip(p, 1e-09, 1.0))
    nn, tt, ss = logp.shape
    dp = np.full((nn, ss), -np.inf)
    dp[:, 0] = logp[:, 0, 0]
    back = np.zeros((nn, tt, ss), np.int8)
    for k in range(1, tt):
        nxt = np.full_like(dp, -np.inf)
        for state in range(ss):
            vs = dp[:, state] + stay_bias
            va = dp[:, state - 1] if state > 0 else np.full(nn, -np.inf)
            keep = vs >= va
            nxt[:, state] = np.where(keep, vs, va) + logp[:, k, state]
            back[:, k, state] = np.where(keep, state, state - 1)
        dp = nxt
    states = np.full(nn, ss - 1, dtype=int)
    path = np.zeros((nn, tt), np.int8)
    ii = np.arange(nn)
    for k in range(tt - 1, -1, -1):
        path[:, k] = states
        if k > 0:
            states = back[ii, k, states]
    return path

def hard_state_features(p, r):
    nn, tt, ss = p.shape
    path = fixed_order_path(p)
    occ = p.mean(axis=1)
    mx = p.max(axis=1)
    centres = (p * GRID[None, :, None]).sum(axis=1) / np.maximum(p.sum(axis=1), 1e-12)
    summaries = np.stack((occ, mx, centres), axis=2).reshape(nn, 15)
    dwell = np.column_stack([(path == state).mean(axis=1) for state in range(5)])
    oi = np.argmax(path == 1, axis=1)
    pi = np.argmax(np.where(path == 2, p[:, :, 2], -np.inf), axis=1)
    fi = np.argmax(path == 4, axis=1)
    ii = np.arange(nn)
    times = np.column_stack((GRID[oi], GRID[pi], GRID[fi]))
    conf = np.column_stack((p[ii, oi, 1], p[ii, pi, 2], p[ii, fi, 4]))
    ent = -(p * np.log(np.clip(p, 1e-09, 1))).sum(axis=2)
    return np.column_stack((summaries, dwell, times, conf, ent.mean(axis=1), ent.max(axis=1), r, (p[:, :, 1] + p[:, :, 2]).mean(axis=1), (p[:, :, 3] + p[:, :, 4]).mean(axis=1)))

def local_profiles(x, probabilities, anchors, response_prob, radius):
    require_predictors_only(x)
    F = finite_model_array(x['state_feat'])
    P = np.asarray(probabilities, float)
    nn = len(F)
    rel = np.linspace(-float(radius), float(radius), 9)
    out = []
    for b in range(3):
        rows = []
        for i in range(nn):
            centre = float(anchors[i, b])
            t = centre + rel
            f = np.column_stack([np.interp(t, GRID, F[i, :, c]) for c in range(F.shape[2])])
            p = np.column_stack([np.interp(t, GRID, P[i, :, c]) for c in range(P.shape[2])])
            seen = ((t >= 0) & (t <= GRID_END)).astype(float)
            context = np.r_[anchors[i], response_prob[i], anchors[i, 1] - anchors[i, 0], anchors[i, 2] - anchors[i, 1]]
            rows.append(np.r_[f.ravel(), p.ravel(), seen, context])
        out.append(np.asarray(rows, float))
    return out

def make_prediction_threshold(_n4, raw, prob, threshold):
    raw = np.asarray(raw, float)
    prob = np.asarray(prob, float)
    on, pk, off = enforce_geometry(raw[:, 0], raw[:, 1], raw[:, 2])
    miss = prob < float(threshold)
    on = on.astype(float)
    pk = pk.astype(float)
    off = off.astype(float)
    on[miss] = np.nan
    pk[miss] = np.nan
    off[miss] = np.nan
    return pd.DataFrame({'pred_response': (~miss).astype(int), 'pred_onset': on, 'pred_peak': pk, 'pred_offset': off})

hb = SimpleNamespace(GRID=GRID, RANDOM_SEED=BASE_MODEL_RANDOM_SEED, finite_model_array=finite_model_array, predictor_view=predictor_view, require_predictors_only=require_predictors_only, grouped_folds=grouped_folds, subset=subset, training_rows=training_rows, FitAudit=FitAudit, StableScaler=StableScaler, hard_state_features=hard_state_features, change_branch_features=change_branch_features, enforce_geometry=enforce_geometry, local_profiles=local_profiles)

@dataclass
class CoherentStateModel:
    scaler: Any
    clf: Any

def fit_coherent_state(hb, data, state_C, audit, stage_prefix):
    audit.record(data, stage_prefix + '_state_encoder')
    F = hb.finite_model_array(data['state_feat']).astype(float, copy=False)
    X = F.reshape(-1, F.shape[-1])
    y = np.asarray(data['states'], int).reshape(-1)
    w = np.asarray(data['weight'], float)
    fw = np.repeat(w, F.shape[1])
    fw = fw / np.mean(fw)
    if len(X) > 60000:
        idx = np.random.default_rng(stable_seed('COHERENT_STATE_SUBSAMPLE', stage_prefix, *sorted(set(map(str, data['pid']))), float(state_C))).choice(len(X), 60000, replace=False)
        X = X[idx]
        y = y[idx]
        fw = fw[idx]
    scaler = hb.StableScaler().fit(X, sample_weight=fw)
    clf = LogisticRegression(C=float(state_C), class_weight='balanced', max_iter=800, random_state=hb.RANDOM_SEED)
    clf.fit(scaler.transform(X), y, sample_weight=fw)
    return CoherentStateModel(scaler=scaler, clf=clf)

def coherent_state_probabilities(hb, model, x):
    hb.require_predictors_only(x)
    F = hb.finite_model_array(x['state_feat']).astype(float, copy=False)
    nn, tt, dd = F.shape
    p0 = model.clf.predict_proba(model.scaler.transform(F.reshape(-1, dd)))
    P = np.full((nn * tt, 5), 1e-08, float)
    for j, cls in enumerate(model.clf.classes_):
        P[:, int(cls)] = p0[:, j]
    P /= np.maximum(P.sum(axis=1, keepdims=True), 1e-12)
    return P.reshape(nn, tt, 5)

def coherent_state_features(hb, P):
    zero_response = np.zeros(len(P), float)
    full = hb.hard_state_features(P, zero_response)
    if full.shape[1] != 31:
        raise RuntimeError(f'Unexpected hard_state_features width {full.shape[1]} (expected 31).')
    return np.column_stack([full[:, :28], full[:, 29:]])

def coherent_ab_features(hb, state_model, x, change_frac):
    P = coherent_state_probabilities(hb, state_model, x)
    SX = coherent_state_features(hb, P)
    CX = hb.change_branch_features(x, float(change_frac))
    return (hb.finite_model_array(np.column_stack([SX, CX])), P)

@dataclass
class CoherentTimingHeads:
    scaler: Any
    regs: Dict[str, Any]
    names: Tuple[str, ...]

    def raw(self, X):
        Z = self.scaler.transform(X)
        return {name: self.regs[name].predict(Z) for name in self.names}

def fit_coherent_timing_heads(hb, X, data, alpha, names, audit, stage_prefix):
    audit.record(data, stage_prefix + '_timing_heads')
    names = tuple(names)
    w = np.asarray(data['weight'], float)
    scaler = hb.StableScaler().fit(X, sample_weight=w)
    Z = scaler.transform(X)
    present = np.asarray(data['present'], int) == 1
    censored = np.asarray(data['censored'], int) == 1
    regs = {}
    for name in names:
        y = np.asarray(data[name], float)
        mask = present & np.isfinite(y)
        if name == 'offset':
            mask &= ~censored
        if int(np.sum(mask)) < 15:
            raise RuntimeError(f'{stage_prefix}: too few rows for {name}.')
        rw = w[mask] / np.mean(w[mask])
        regs[name] = Ridge(alpha=float(alpha)).fit(Z[mask], y[mask], sample_weight=rw)
    return CoherentTimingHeads(scaler=scaler, regs=regs, names=names)

@dataclass
class CoherentABModel:
    hb: Any
    state: CoherentStateModel
    heads: CoherentTimingHeads
    change_frac: float

    def raw(self, x):
        X, P = coherent_ab_features(self.hb, self.state, x, self.change_frac)
        pred = self.heads.raw(X)
        on, pk, off = self.hb.enforce_geometry(np.asarray(pred['onset'], float), np.asarray(pred['peak'], float), np.asarray(pred['offset'], float))
        return (np.column_stack([on, pk, off]), P, X)

def fit_coherent_AB(hb, data, cfg, audit, stage_prefix='AB'):
    state = fit_coherent_state(hb, data, float(cfg['state_C']), audit, stage_prefix)
    X, _ = coherent_ab_features(hb, state, hb.predictor_view(data), float(cfg['change_frac']))
    heads = fit_coherent_timing_heads(hb, X, data, cfg['timing_alpha'], ('onset', 'peak', 'offset'), audit, stage_prefix)
    return CoherentABModel(hb=hb, state=state, heads=heads, change_frac=float(cfg['change_frac']))

@dataclass
class CoherentABCModel:
    hb: Any
    ab: CoherentABModel
    response_scaler: Any
    response_clf: Any
    response_threshold: float

    def raw(self, x):
        raw, P, X = self.ab.raw(x)
        prob = self.response_clf.predict_proba(self.response_scaler.transform(X))[:, 1]
        return (raw, np.asarray(prob, float), P)

    def predict(self, x):
        raw, prob, _ = self.raw(x)
        return make_prediction_threshold(self.hb, raw, prob, self.response_threshold)

def fit_coherent_ABC(hb, data, cfg, audit, stage_prefix='ABC'):
    ab = fit_coherent_AB(hb, data, cfg, audit, stage_prefix=stage_prefix + '_AB')
    X, _ = coherent_ab_features(hb, ab.state, hb.predictor_view(data), ab.change_frac)
    audit.record(data, stage_prefix + '_response_detector')
    w = np.asarray(data['weight'], float)
    y = np.asarray(data['present'], int)
    response_scaler = hb.StableScaler().fit(X, sample_weight=w)
    response_clf = LogisticRegression(C=0.5, class_weight='balanced', max_iter=1000, random_state=hb.RANDOM_SEED)
    response_clf.fit(response_scaler.transform(X), y, sample_weight=w / np.mean(w))
    return CoherentABCModel(hb=hb, ab=ab, response_scaler=response_scaler, response_clf=response_clf, response_threshold=float(cfg['response_threshold']))

def coherent_oof_anchors(hb, data, abc_cfg, outer_forbidden, audit_parent, context):
    pids = sorted(set(map(str, data['pid'])))
    nn = len(data['pid'])
    raw = np.zeros((nn, 3), float)
    rp = np.zeros(nn, float)
    P = np.zeros((nn, len(hb.GRID), 5), float)
    covered = np.zeros(nn, int)
    folds = hb.grouped_folds(pids, stable_seed('COHERENT_LBR_OOF', context, canonical_cfg(abc_cfg), *pids), LOCAL_OOF_FOLDS_COHERENT)
    for held in folds:
        mask = np.isin(data['pid'], held)
        tr = hb.subset(data, ~mask)
        va = hb.subset(data, mask)
        tr_pids = sorted(set(map(str, tr['pid'])))
        audit = hb.FitAudit(tr_pids, set(outer_forbidden) | set(held), context + '/LBR_OOF')
        model = fit_coherent_ABC(hb, tr, abc_cfg, audit, stage_prefix='LBR_OOF_ABC')
        r, p, pp = model.raw(hb.predictor_view(va))
        raw[mask] = r
        rp[mask] = p
        P[mask] = pp
        covered[mask] += 1
        audit_parent.rows.extend(audit.rows)
    if not np.all(covered == 1):
        raise RuntimeError('Coherent LBR OOF anchor coverage failed.')
    return (raw, rp, P)


In [ ]:
# Cell 2: Load saved seed outputs and define component neutralization
from __future__ import annotations

@dataclass
class CoherentFullModel:
    hb: Any
    abc: CoherentABCModel
    local_cfg: Dict[str, float]
    scalers: List[Any]
    regs: List[Any]

    def refined_raw(self, x, coarse_raw=None, rp=None, P=None):
        if coarse_raw is None or rp is None or P is None:
            coarse_raw, rp, P = self.abc.raw(x)
        coarse_raw = np.asarray(coarse_raw, float).copy()
        on, pk, off = self.hb.enforce_geometry(coarse_raw[:, 0], coarse_raw[:, 1], coarse_raw[:, 2])
        coarse_raw = np.column_stack([on, pk, off])
        profiles = self.hb.local_profiles(x, P, coarse_raw, rp, float(self.local_cfg['radius']))
        refined = np.asarray(coarse_raw, float).copy()
        for b in range(3):
            delta = self.regs[b].predict(self.scalers[b].transform(profiles[b]))
            radius = float(self.local_cfg['radius'])
            refined[:, b] += float(self.local_cfg['strength']) * np.clip(delta, -radius, radius)
        on, pk, off = self.hb.enforce_geometry(refined[:, 0], refined[:, 1], refined[:, 2])
        return np.column_stack([on, pk, off])

    def predict(self, x):
        raw, rp, P = self.abc.raw(x)
        refined = self.refined_raw(x, raw, rp, P)
        return make_prediction_threshold(self.hb, refined, rp, self.abc.response_threshold)

def fit_coherent_full(hb, data, abc_cfg, local_cfg, audit, outer_forbidden, context):
    abc = fit_coherent_ABC(hb, data, abc_cfg, audit, stage_prefix='FULL_ABC')
    raw_oof, rp_oof, P_oof = coherent_oof_anchors(hb, data, abc_cfg, outer_forbidden, audit, context)
    profiles = hb.local_profiles(hb.predictor_view(data), P_oof, raw_oof, rp_oof, float(local_cfg['radius']))
    audit.record(data, 'FULL_LBR_residual_heads_on_source_OOF_anchors')
    present = np.asarray(data['present'], int) == 1
    censored = np.asarray(data['censored'], int) == 1
    scalers = []
    regs = []
    for b, name in enumerate(('onset', 'peak', 'offset')):
        y = np.asarray(data[name], float)
        mask = present & np.isfinite(y)
        if name == 'offset':
            mask &= ~censored
        scaler = hb.StableScaler().fit(profiles[b][mask])
        residual = y[mask] - raw_oof[mask, b]
        reg = Ridge(alpha=float(local_cfg['local_alpha'])).fit(scaler.transform(profiles[b][mask]), residual)
        scalers.append(scaler)
        regs.append(reg)
    return CoherentFullModel(hb=hb, abc=abc, local_cfg=dict(local_cfg), scalers=scalers, regs=regs)

def seed_cache_dir(seed: int) -> Path:
    return SHIN_ROOT / 'analysis/HemoBoundary_Main' / f'seed_{int(seed):02d}' / 'fresh_cache'

def seed_selected_table(seed: int) -> Path:
    return SHIN_ROOT / 'analysis/HemoBoundary_Main' / f'seed_{int(seed):02d}' / 'tables' / 'Table3_Selected_Configurations.csv'

def load_seed_caches(seed: int):
    cache_dir = seed_cache_dir(seed)
    if not cache_dir.is_dir():
        raise FileNotFoundError(f'Expected completed seed cache does not exist:\n{cache_dir}')
    caches = {}
    for pid in ALL_PIDS:
        path = cache_dir / f'{pid}.npz'
        if not path.is_file():
            raise FileNotFoundError(f'Missing completed cache: {path}')
        with np.load(path, allow_pickle=False) as z:
            item = {}
            for prefix, split in (('tr', 'train'), ('te', 'test')):
                fields = (*PREDICTOR_KEYS, *LABEL_KEYS)
                missing = [k for k in fields if f'{prefix}_{k}' not in z.files]
                if missing:
                    raise RuntimeError(f'{path.name}/{split} missing {missing}')
                d = {k: np.array(z[f'{prefix}_{k}'], copy=True) for k in fields}
                nn = len(d['present'])
                if d['state_feat'].shape != (nn, len(GRID), 9):
                    raise RuntimeError(f"{path.name}/{split}: state feature shape mismatch {d['state_feat'].shape}")
                for key in PREDICTOR_KEYS:
                    d[key] = finite_model_array(d[key])
                d['pid'] = np.full(nn, pid, dtype='U5')
                d['weight'] = np.ones(nn, float)
                item[split] = d
            caches[pid] = item
    return (caches, cache_dir)

def normalize_pid(x) -> str:
    s = str(x).strip().upper()
    if s.startswith('VP'):
        digits = ''.join((c for c in s if c.isdigit()))
        if digits:
            return f'VP{int(digits):03d}'
    digits = ''.join((c for c in s if c.isdigit()))
    return f'VP{int(digits):03d}' if digits else s

def parse_json_cfg(x):
    if isinstance(x, dict):
        return dict(x)
    s = str(x).strip()
    if not s:
        return None
    return {k: float(v) for k, v in json.loads(s).items()}

def load_full_config(seed: int, pid: str):
    table = seed_selected_table(seed)
    if not table.is_file():
        raise FileNotFoundError(f'Selected-configuration table does not exist:\n{table}')
    q = pd.read_csv(table)
    pid_col = 'participant' if 'participant' in q.columns else 'outer_target' if 'outer_target' in q.columns else None
    if pid_col is None:
        raise RuntimeError(f'Selected-config table has no participant column: {list(q.columns)}')
    if 'row' not in q.columns:
        raise RuntimeError(f'Selected-config table has no row column: {list(q.columns)}')
    pids = q[pid_col].map(normalize_pid)
    full = q[(pids == pid) & (q['row'].astype(str) == ROW_FULL)].copy()
    if len(full) != 1:
        raise RuntimeError(f'Expected exactly one full configuration for seed={seed}, participant={pid}; found {len(full)}.')
    row = full.iloc[0]
    abc_cfg = parse_json_cfg(row['global_cfg'])
    local_cfg = parse_json_cfg(row['local_cfg'])
    expected_global = {'state_C', 'change_frac', 'timing_alpha', 'response_threshold'}
    expected_local = {'radius', 'local_alpha', 'strength'}
    if abc_cfg is None or set(abc_cfg) != expected_global:
        raise RuntimeError(f'Unexpected ABC config for seed={seed}, {pid}: {abc_cfg}')
    if local_cfg is None or set(local_cfg) != expected_local:
        raise RuntimeError(f'Unexpected LBR config for seed={seed}, {pid}: {local_cfg}')
    return (abc_cfg, local_cfg, table)

def source_neutral_values(train: Mapping[str, np.ndarray]):
    present = np.asarray(train['present'], int) == 1
    censored = np.asarray(train['censored'], int) == 1
    onset = np.asarray(train['onset'], float)
    peak = np.asarray(train['peak'], float)
    offset = np.asarray(train['offset'], float)
    m_on = present & np.isfinite(onset)
    m_pk = present & np.isfinite(peak)
    m_off = present & ~censored & np.isfinite(offset)
    return {'onset': float(np.median(onset[m_on])), 'peak': float(np.median(peak[m_pk])), 'offset': float(np.median(offset[m_off]))}

def make_all_pass_prediction(refined_raw, response_prob):
    refined_raw = np.asarray(refined_raw, float)
    on, pk, off = enforce_geometry(refined_raw[:, 0], refined_raw[:, 1], refined_raw[:, 2])
    return pd.DataFrame({'pred_response': np.ones(len(refined_raw), dtype=int), 'pred_onset': on, 'pred_peak': pk, 'pred_offset': off})

def functional_predictions(model: CoherentFullModel, x, train):
    coarse, rp, P = model.abc.raw(x)
    full_refined = model.refined_raw(x, coarse, rp, P)
    full = make_prediction_threshold(hb, full_refined, rp, model.abc.response_threshold)
    no_lbr = make_prediction_threshold(hb, coarse, rp, model.abc.response_threshold)
    no_c = make_all_pass_prediction(full_refined, rp)
    neutral = source_neutral_values(train)
    raw_a = np.asarray(coarse, float).copy()
    raw_a[:, 0] = neutral['onset']
    raw_a[:, 1] = neutral['peak']
    a_refined = model.refined_raw(x, raw_a, rp, P)
    no_a = make_prediction_threshold(hb, a_refined, rp, model.abc.response_threshold)
    raw_b = np.asarray(coarse, float).copy()
    raw_b[:, 2] = neutral['offset']
    b_refined = model.refined_raw(x, raw_b, rp, P)
    no_b = make_prediction_threshold(hb, b_refined, rp, model.abc.response_threshold)
    return ({'FULL': full, 'A_OUTPUT_NEUTRALIZED': no_a, 'B_OUTPUT_NEUTRALIZED': no_b, 'C_GATE_NEUTRALIZED': no_c, 'LBR_NEUTRALIZED': no_lbr}, neutral, rp, coarse, full_refined)

def balanced_accuracy(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, int)
    p = np.asarray(p, int)
    pos = y == 1
    neg = y == 0
    sens = np.mean(p[pos] == 1) if np.any(pos) else np.nan
    spec = np.mean(p[neg] == 0) if np.any(neg) else np.nan
    return float(np.nanmean([sens, spec]))

def participant_metrics(truth, pred):
    present = np.asarray(truth['present'], int) == 1
    censored = np.asarray(truth['censored'], int) == 1
    decision = pred['pred_response'].to_numpy(int)
    ton = np.asarray(truth['onset'], float)
    tpk = np.asarray(truth['peak'], float)
    toff = np.asarray(truth['offset'], float)
    pon = pred['pred_onset'].to_numpy(float)
    ppk = pred['pred_peak'].to_numpy(float)
    poff = pred['pred_offset'].to_numpy(float)
    out = {'response_balanced_accuracy': balanced_accuracy(np.asarray(truth['present'], int), decision)}
    conditional = {}
    for name, tt, pp, exact in (('onset', ton, pon, present & np.isfinite(ton)), ('peak', tpk, ppk, present & np.isfinite(tpk)), ('offset', toff, poff, present & ~censored & np.isfinite(toff))):
        m = exact & (decision == 1) & np.isfinite(pp)
        conditional[f'conditional_{name}_mae_sec'] = float(np.mean(np.abs(pp[m] - tt[m]))) if np.any(m) else np.nan
        conditional[f'conditional_{name}_localized_n'] = int(np.sum(m))
    dur_true = toff - ton
    dur_pred = poff - pon
    exact_dur = present & ~censored & np.isfinite(dur_true)
    m = exact_dur & (decision == 1) & np.isfinite(dur_pred)
    conditional['conditional_duration_mae_sec'] = float(np.mean(np.abs(dur_pred[m] - dur_true[m]))) if np.any(m) else np.nan
    conditional['conditional_duration_localized_n'] = int(np.sum(m))
    out.update(conditional)
    e2e_values = {}
    for name, tt, pp, exact in (('onset', ton, pon, present & np.isfinite(ton)), ('peak', tpk, ppk, present & np.isfinite(tpk)), ('offset', toff, poff, present & ~censored & np.isfinite(toff))):
        idx = np.flatnonzero(exact)
        err = np.full(len(idx), MISS_PENALTY_SEC, float)
        if len(idx):
            good = (decision[idx] == 1) & np.isfinite(pp[idx])
            err[good] = np.abs(pp[idx][good] - tt[idx][good])
        value = float(np.mean(err)) if len(err) else np.nan
        out[f'e2e_{name}_mae_sec'] = value
        e2e_values[name] = value
    idx = np.flatnonzero(exact_dur)
    err = np.full(len(idx), MISS_PENALTY_SEC, float)
    if len(idx):
        good = (decision[idx] == 1) & np.isfinite(dur_pred[idx])
        err[good] = np.abs(dur_pred[idx][good] - dur_true[idx][good])
    value = float(np.mean(err)) if len(err) else np.nan
    out['e2e_duration_mae_sec'] = value
    e2e_values['duration'] = value
    vals = np.asarray([e2e_values['onset'], e2e_values['peak'], e2e_values['offset'], e2e_values['duration']], float)
    scales = np.asarray([1.0, 1.0, 1.5, 1.5], float)
    out['e2e_selection_loss'] = float(np.nanmean(vals / scales))
    out['predicted_response_rate'] = float(np.mean(decision == 1))
    out['n_positive'] = int(np.sum(present))
    out['n_negative'] = int(np.sum(~present))
    return out

def per_trial_frame(seed, pid, variant, truth, pred, response_prob, neutral, abc_cfg, local_cfg):
    return pd.DataFrame({'repeat_seed': int(seed), 'participant': pid, 'variant': variant, 'trial_index': np.arange(len(pred), dtype=int), 'truth_present': np.asarray(truth['present'], int), 'truth_onset': np.asarray(truth['onset'], float), 'truth_peak': np.asarray(truth['peak'], float), 'truth_offset': np.asarray(truth['offset'], float), 'truth_censored': np.asarray(truth['censored'], int), 'pred_response': pred['pred_response'].to_numpy(int), 'response_probability_from_C': np.asarray(response_prob, float), 'pred_onset': pred['pred_onset'].to_numpy(float), 'pred_peak': pred['pred_peak'].to_numpy(float), 'pred_offset': pred['pred_offset'].to_numpy(float), 'source_neutral_onset': float(neutral['onset']), 'source_neutral_peak': float(neutral['peak']), 'source_neutral_offset': float(neutral['offset']), 'selected_abc_cfg': canonical_cfg(abc_cfg), 'selected_lbr_cfg': canonical_cfg(local_cfg)})

def run_validation_checks():
    assert len(ALL_PIDS) == 26
    assert len(GRID) == 41
    assert set(SEEDS).issubset(set(range(1, 11)))
    assert len(set(SEEDS)) == len(SEEDS)
    dummy_raw = np.asarray([[1.0, 4.0, 10.0], [2.0, 5.0, 12.0]])
    dummy_prob = np.asarray([0.9, 0.1])
    pred = make_prediction_threshold(hb, dummy_raw, dummy_prob, 0.5)
    assert pred['pred_response'].tolist() == [1, 0]
    assert np.isfinite(pred.loc[0, 'pred_onset'])
    assert np.isnan(pred.loc[1, 'pred_onset'])
    print('Functional necessity validation checks: PASS')


In [ ]:
# Cell 3: Run functional-necessity analysis for seeds 1–10
from __future__ import annotations

def run_all_functional_necessity_seeds():
    mount_drive()
    run_validation_checks()
    if not SHIN_ROOT.is_dir():
        raise FileNotFoundError(f'Expected SHIN project folder does not exist:\n{SHIN_ROOT}')
    for sub in ('checkpoints', 'results', 'audit'):
        (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
    print('=' * 180)
    print('HEMOBOUNDARY — FUNCTIONAL NECESSITY ANALYSIS')
    print('=' * 180)
    print('Assigned seeds                 :', list(SEEDS))
    print('Participants per seed          : 26')
    print('Nested hyperparameter search   : NO')
    print('External core .py required     : NO')
    print('Source-code Drive search       : NO')
    print('Seed-specific completed banks  : YES')
    print('Seed/participant selected cfg   : YES')
    print('Output                         :', OUT_DIR)
    print()
    all_metric_parts = []
    all_trial_parts = []
    all_audit_parts = []
    for seed in SEEDS:
        print('\n' + '=' * 180)
        print(f'SEED {seed:02d}')
        print('=' * 180)
        configure_seed(seed)
        caches, cache_dir = load_seed_caches(seed)
        print('Cache:', cache_dir)
        print('Config table:', seed_selected_table(seed))
        for index, pid in enumerate(ALL_PIDS, 1):
            cp_metrics = OUT_DIR / 'checkpoints' / f'seed_{seed:02d}_{pid}_METRICS.csv'
            cp_trials = OUT_DIR / 'checkpoints' / f'seed_{seed:02d}_{pid}_TRIALS.csv.gz'
            cp_audit = OUT_DIR / 'checkpoints' / f'seed_{seed:02d}_{pid}_AUDIT.csv'
            if cp_metrics.is_file() and cp_trials.is_file() and cp_audit.is_file():
                print(f'[seed {seed:02d}] [{index:02d}/26] {pid}: checkpoint')
                all_metric_parts.append(pd.read_csv(cp_metrics))
                all_trial_parts.append(pd.read_csv(cp_trials))
                all_audit_parts.append(pd.read_csv(cp_audit))
                continue
            abc_cfg, local_cfg, cfg_table = load_full_config(seed, pid)
            source_pids = [p for p in ALL_PIDS if p != pid]
            train = training_rows(caches, source_pids)
            truth = caches[pid]['test']
            x = predictor_view(truth)
            audit = FitAudit(source_pids, [pid], f'FUNCTIONAL_NECESSITY/seed={seed:02d}/outer={pid}')
            print(f'[seed {seed:02d}] [{index:02d}/26] {pid}: fitting the fixed source model', flush=True)
            model = fit_coherent_full(hb, train, abc_cfg, local_cfg, audit, outer_forbidden={pid}, context=f'FUNCTIONAL_NECESSITY/seed={seed:02d}/outer={pid}')
            variants, neutral, rp, coarse, full_refined = functional_predictions(model, x, train)
            native_full = model.predict(x)
            for col in ('pred_onset', 'pred_peak', 'pred_offset'):
                a = native_full[col].to_numpy(float)
                b = variants['FULL'][col].to_numpy(float)
                if not np.allclose(a, b, atol=1e-10, rtol=0.0, equal_nan=True):
                    raise RuntimeError(f'FULL parity failed for {pid}/{col}.')
            if not np.array_equal(native_full['pred_response'].to_numpy(int), variants['FULL']['pred_response'].to_numpy(int)):
                raise RuntimeError(f'FULL response parity failed for {pid}.')
            metric_rows = []
            trial_rows = []
            for variant, pred in variants.items():
                mm = participant_metrics(truth, pred)
                metric_rows.append({'repeat_seed': int(seed), 'participant': pid, 'variant': variant, **mm, 'source_neutral_onset': float(neutral['onset']), 'source_neutral_peak': float(neutral['peak']), 'source_neutral_offset': float(neutral['offset']), 'selected_abc_cfg': canonical_cfg(abc_cfg), 'selected_lbr_cfg': canonical_cfg(local_cfg)})
                trial_rows.append(per_trial_frame(seed, pid, variant, truth, pred, rp, neutral, abc_cfg, local_cfg))
            metric_df = pd.DataFrame(metric_rows)
            trial_df = pd.concat(trial_rows, ignore_index=True)
            audit_df = pd.DataFrame(audit.rows)
            audit_df.insert(0, 'repeat_seed', int(seed))
            audit_df.insert(1, 'outer_participant', pid)
            audit_df['selected_abc_cfg'] = canonical_cfg(abc_cfg)
            audit_df['selected_lbr_cfg'] = canonical_cfg(local_cfg)
            audit_df['selected_config_source'] = str(cfg_table)
            audit_df['seed_cache_source'] = str(cache_dir)
            atomic_csv(metric_df, cp_metrics)
            atomic_csv(trial_df, cp_trials)
            atomic_csv(audit_df, cp_audit)
            all_metric_parts.append(metric_df)
            all_trial_parts.append(trial_df)
            all_audit_parts.append(audit_df)
            del (model, train, truth, x, variants, trial_rows, trial_df)
            gc.collect()
        del caches
        gc.collect()
    metrics = pd.concat(all_metric_parts, ignore_index=True)
    trials = pd.concat(all_trial_parts, ignore_index=True)
    audits = pd.concat(all_audit_parts, ignore_index=True)
    atomic_csv(metrics, OUT_DIR / 'results' / 'PARTICIPANT_METRICS.csv')
    atomic_csv(trials, OUT_DIR / 'results' / 'PER_TRIAL_PREDICTIONS.csv.gz')
    atomic_csv(audits, OUT_DIR / 'audit' / 'Fit_Membership_Check.csv.gz')
    summary = metrics.groupby('variant', as_index=False).agg(rows=('participant', 'size'), participants=('participant', 'nunique'), seeds=('repeat_seed', 'nunique'), response_BA=('response_balanced_accuracy', 'mean'), onset_MAE=('e2e_onset_mae_sec', 'mean'), peak_MAE=('e2e_peak_mae_sec', 'mean'), offset_MAE=('e2e_offset_mae_sec', 'mean'), duration_MAE=('e2e_duration_mae_sec', 'mean'), selection_loss=('e2e_selection_loss', 'mean')).sort_values('variant').reset_index(drop=True)
    atomic_csv(summary, OUT_DIR / 'results' / 'Seed_Summary.csv')
    atomic_json({'analysis': 'Functional necessity through output intervention', 'seed_output_dir': SEED_OUTPUT_DIRNAME, 'seeds': list(SEEDS), 'participants_per_seed': 26, 'variants': ['FULL', 'A_OUTPUT_NEUTRALIZED', 'B_OUTPUT_NEUTRALIZED', 'C_GATE_NEUTRALIZED', 'LBR_NEUTRALIZED'], 'nested_hyperparameter_search': False, 'external_core_file_required': False, 'source_code_search': False, 'seed_specific_existing_caches': True, 'outer_target_used_for_fitting': False, 'neutral_values_source': "25 outer-source participants' TRAIN banks only", 'interpretation': 'functional output intervention; not a fully factorial architecture independence experiment'}, OUT_DIR / 'audit' / 'RUN_MANIFEST.json')
    print('\n' + '=' * 180)
    print('Analysis complete')
    print('=' * 180)
    print(summary.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    print()
    print('Saved:', OUT_DIR)


In [ ]:
# Cell 4: Merge all 10 seeds and perform participant-level inference
from __future__ import annotations

from pathlib import Path

from typing import List, Tuple

import numpy as np

import pandas as pd

DRIVE_ROOT = Path('/content/gdrive/MyDrive')

SHIN_ROOT = DRIVE_ROOT / 'HonorProject' / 'HemoBoundary' / 'SHIN'

ANALYSIS_ROOT = SHIN_ROOT / 'analysis/Functional_Necessity'

COMBINED_OUTPUT = ANALYSIS_ROOT / 'combined'

SEED_OUTPUT = ANALYSIS_ROOT / 'all_seeds_01_10'

SEED_TO_DIRECTORY = {seed: SEED_OUTPUT for seed in range(1, 11)}

ALL_SEEDS = tuple(range(1, 11))

ALL_PIDS = tuple((f'VP{i:03d}' for i in range(1, 27)))

VARIANTS = ('FULL', 'A_OUTPUT_NEUTRALIZED', 'B_OUTPUT_NEUTRALIZED', 'C_GATE_NEUTRALIZED', 'LBR_NEUTRALIZED')

COMPONENT = {'A_OUTPUT_NEUTRALIZED': 'A — Temporal Boundary Estimation', 'B_OUTPUT_NEUTRALIZED': 'B — Recovery Estimation', 'C_GATE_NEUTRALIZED': 'C — Response Detection', 'LBR_NEUTRALIZED': 'LBR — Local Boundary Refinement'}

METRICS = ('response_balanced_accuracy', 'e2e_onset_mae_sec', 'e2e_peak_mae_sec', 'e2e_offset_mae_sec', 'e2e_duration_mae_sec', 'e2e_selection_loss')

METRIC_LABEL = {'response_balanced_accuracy': 'Response BA', 'e2e_onset_mae_sec': 'Onset MAE (s)', 'e2e_peak_mae_sec': 'Peak MAE (s)', 'e2e_offset_mae_sec': 'Offset MAE (s)', 'e2e_duration_mae_sec': 'Duration MAE (s)', 'e2e_selection_loss': 'Selection loss'}

LOWER_IS_BETTER = {'response_balanced_accuracy': False, 'e2e_onset_mae_sec': True, 'e2e_peak_mae_sec': True, 'e2e_offset_mae_sec': True, 'e2e_duration_mae_sec': True, 'e2e_selection_loss': True}

BOOT_REPS = 20000

SIGNFLIP_REPS = 20000

TOL = 1e-12

def mount_drive():
    if DRIVE_ROOT.is_dir():
        return
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=False)

def save_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    compression = 'gzip' if str(path).endswith('.gz') else None
    df.to_csv(tmp, index=False, compression=compression)
    tmp.replace(path)

def metric_checkpoint(seed: int, pid: str) -> Path:
    return SEED_TO_DIRECTORY[seed] / 'checkpoints' / f'seed_{seed:02d}_{pid}_METRICS.csv'

def trial_checkpoint(seed: int, pid: str) -> Path:
    return SEED_TO_DIRECTORY[seed] / 'checkpoints' / f'seed_{seed:02d}_{pid}_TRIALS.csv.gz'

def bootstrap_ci(x: np.ndarray, seed: int) -> Tuple[float, float, float]:
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    mean = float(np.mean(x))
    if len(x) == 1:
        return (mean, mean, mean)
    rng = np.random.default_rng(seed)
    sims = np.empty(BOOT_REPS, float)
    done = 0
    while done < BOOT_REPS:
        n = min(2000, BOOT_REPS - done)
        ix = rng.integers(0, len(x), size=(n, len(x)))
        sims[done:done + n] = np.mean(x[ix], axis=1)
        done += n
    lo, hi = np.quantile(sims, [0.025, 0.975])
    return (mean, float(lo), float(hi))

def signflip_p(x: np.ndarray, seed: int) -> float:
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    obs = abs(float(np.mean(x)))
    if obs <= 1e-15:
        return 1.0
    rng = np.random.default_rng(seed)
    hits = 0
    done = 0
    while done < SIGNFLIP_REPS:
        n = min(2000, SIGNFLIP_REPS - done)
        signs = rng.choice((-1.0, 1.0), size=(n, len(x)))
        stats = np.abs(np.mean(signs * x[None, :], axis=1))
        hits += int(np.sum(stats >= obs - 1e-15))
        done += n
    return float((hits + 1) / (SIGNFLIP_REPS + 1))

def holm_adjust(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, float)
    out = np.full(len(p), np.nan, float)
    valid = np.flatnonzero(np.isfinite(p))
    if len(valid) == 0:
        return out
    order = valid[np.argsort(p[valid])]
    running = 0.0
    m = len(order)
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        out[idx] = min(1.0, running)
    return out

def full_better_gain(full: np.ndarray, neutral: np.ndarray, metric: str) -> np.ndarray:
    full = np.asarray(full, float)
    neutral = np.asarray(neutral, float)
    if LOWER_IS_BETTER[metric]:
        return neutral - full
    return full - neutral

def load_metrics():
    parts: List[pd.DataFrame] = []
    audit = []
    for seed in ALL_SEEDS:
        for pid in ALL_PIDS:
            p = metric_checkpoint(seed, pid)
            ok = p.is_file()
            audit.append({'repeat_seed': seed, 'participant': pid, 'metrics_exists': ok, 'metrics_file': str(p)})
            if not ok:
                continue
            q = pd.read_csv(p)
            if 'repeat_seed' not in q.columns:
                q.insert(0, 'repeat_seed', seed)
            if 'participant' not in q.columns:
                q.insert(1, 'participant', pid)
            parts.append(q)
    return (pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(), pd.DataFrame(audit))

def validate_metrics(metrics: pd.DataFrame, audit: pd.DataFrame):
    missing = audit[~audit['metrics_exists']].copy()
    if len(missing):
        print('\nMissing or incomplete checkpoints')
        print(missing[['repeat_seed', 'participant']].to_string(index=False))
        raise RuntimeError(f'Jobs are not finished: {len(missing)} of 260 seed-participant metric checkpoints are missing.')
    required = {'repeat_seed', 'participant', 'variant', *METRICS}
    missing_cols = required - set(metrics.columns)
    if missing_cols:
        raise RuntimeError(f'Missing metric columns: {sorted(missing_cols)}')
    metrics['repeat_seed'] = pd.to_numeric(metrics['repeat_seed'], errors='raise').astype(int)
    metrics['participant'] = metrics['participant'].astype(str)
    metrics['variant'] = metrics['variant'].astype(str)
    dup = metrics.duplicated(['repeat_seed', 'participant', 'variant'], keep=False)
    if dup.any():
        raise RuntimeError('Duplicate seed-participant-variant rows:\n' + metrics.loc[dup, ['repeat_seed', 'participant', 'variant']].to_string(index=False))
    expected = 10 * 26 * 5
    if len(metrics) != expected:
        raise RuntimeError(f'Expected {expected} metric rows; found {len(metrics)}.')
    if set(metrics['repeat_seed']) != set(ALL_SEEDS):
        raise RuntimeError('Seed coverage is not exactly 1..10.')
    if set(metrics['participant']) != set(ALL_PIDS):
        raise RuntimeError('Participant coverage is not exactly VP001..VP026.')
    if set(metrics['variant']) != set(VARIANTS):
        raise RuntimeError('Variant coverage is not exactly the expected five variants.')
    coverage = metrics.groupby(['repeat_seed', 'participant'])['variant'].nunique()
    if not np.all(coverage.to_numpy() == 5):
        raise RuntimeError('At least one seed-participant is missing one or more variants.')

def load_trials():
    parts: List[pd.DataFrame] = []
    missing = []
    for seed in ALL_SEEDS:
        for pid in ALL_PIDS:
            p = trial_checkpoint(seed, pid)
            if not p.is_file():
                missing.append((seed, pid))
                continue
            q = pd.read_csv(p)
            if 'repeat_seed' not in q.columns:
                q.insert(0, 'repeat_seed', seed)
            if 'participant' not in q.columns:
                q.insert(1, 'participant', pid)
            parts.append(q)
    if missing:
        raise RuntimeError(f'Per-trial merge found {len(missing)} missing files. First missing: {missing[:10]}')
    out = pd.concat(parts, ignore_index=True)
    dup = out.duplicated(['repeat_seed', 'participant', 'variant', 'trial_index'], keep=False)
    if dup.any():
        raise RuntimeError('Duplicate per-trial rows found.')
    counts = out.groupby(['repeat_seed', 'participant', 'variant'])['trial_index'].nunique()
    bad = counts[counts != 36]
    if len(bad):
        raise RuntimeError('Expected exactly 36 test trials per seed-participant-variant.\n' + bad.to_string())
    expected = 10 * 26 * 5 * 36
    if len(out) != expected:
        raise RuntimeError(f'Expected {expected} trial rows; found {len(out)}.')
    return out

def participant_average(metrics: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [c for c in metrics.columns if c not in {'repeat_seed', 'participant', 'variant', 'selected_abc_cfg', 'selected_lbr_cfg'} and pd.api.types.is_numeric_dtype(metrics[c])]
    avg = metrics.groupby(['participant', 'variant'], as_index=False)[numeric_cols].mean()
    nseed = metrics.groupby(['participant', 'variant'], as_index=False)['repeat_seed'].nunique().rename(columns={'repeat_seed': 'n_seeds_averaged'})
    avg = avg.merge(nseed, on=['participant', 'variant'], validate='one_to_one')
    if not np.all(avg['n_seeds_averaged'].to_numpy() == 10):
        raise RuntimeError('Participant averages did not use exactly 10 seeds everywhere.')
    return avg

def means_table(avg: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for variant in VARIANTS:
        q = avg[avg['variant'] == variant]
        row = {'variant': variant, 'component_intervention': 'Intact full model' if variant == 'FULL' else COMPONENT[variant], 'n_participants': int(q['participant'].nunique())}
        for metric in METRICS:
            x = pd.to_numeric(q[metric], errors='coerce').to_numpy(float)
            row[f'{metric}_mean'] = float(np.nanmean(x))
            row[f'{metric}_sd'] = float(np.nanstd(x, ddof=1))
        rows.append(row)
    return pd.DataFrame(rows)

def paired_table(avg: pd.DataFrame) -> pd.DataFrame:
    full = avg[avg['variant'] == 'FULL'].set_index('participant').sort_index()
    rows = []
    for neutral_variant in VARIANTS[1:]:
        neutral = avg[avg['variant'] == neutral_variant].set_index('participant').sort_index()
        common = full.index.intersection(neutral.index)
        if len(common) != 26:
            raise RuntimeError(f'{neutral_variant}: expected 26 paired participants, got {len(common)}.')
        for metric in METRICS:
            a = pd.to_numeric(full.loc[common, metric], errors='coerce').to_numpy(float)
            b = pd.to_numeric(neutral.loc[common, metric], errors='coerce').to_numpy(float)
            ok = np.isfinite(a) & np.isfinite(b)
            a, b = (a[ok], b[ok])
            gain = full_better_gain(a, b, metric)
            sid = VARIANTS.index(neutral_variant) * 100 + METRICS.index(metric)
            mean_gain, lo, hi = bootstrap_ci(gain, 202609260 + sid)
            p = signflip_p(gain, 202609261 + sid)
            rows.append({'component': COMPONENT[neutral_variant], 'neutralized_variant': neutral_variant, 'metric': metric, 'metric_label': METRIC_LABEL[metric], 'n_participants': int(len(gain)), 'full_mean': float(np.mean(a)), 'neutralized_mean': float(np.mean(b)), 'mean_gain_positive_full_better': float(mean_gain), 'ci95_low': float(lo), 'ci95_high': float(hi), 'participants_full_better': int(np.sum(gain > TOL)), 'participants_equal': int(np.sum(np.abs(gain) <= TOL)), 'participants_neutralized_better': int(np.sum(gain < -TOL)), 'raw_signflip_p': float(p)})
    out = pd.DataFrame(rows)
    out['holm_p_within_component_6'] = np.nan
    for component in out['component'].unique():
        ix = out.index[out['component'] == component].tolist()
        out.loc[ix, 'holm_p_within_component_6'] = holm_adjust(out.loc[ix, 'raw_signflip_p'].to_numpy(float))
    out['holm_p_global_24'] = holm_adjust(out['raw_signflip_p'].to_numpy(float))
    return out

def seedwise_table(metrics: pd.DataFrame) -> pd.DataFrame:
    return metrics.groupby(['repeat_seed', 'variant'], as_index=False).agg(n_participants=('participant', 'nunique'), response_BA=('response_balanced_accuracy', 'mean'), onset_MAE=('e2e_onset_mae_sec', 'mean'), peak_MAE=('e2e_peak_mae_sec', 'mean'), offset_MAE=('e2e_offset_mae_sec', 'mean'), duration_MAE=('e2e_duration_mae_sec', 'mean'), selection_loss=('e2e_selection_loss', 'mean')).sort_values(['repeat_seed', 'variant']).reset_index(drop=True)

def compact_means(means: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({'variant': means['variant'], 'BA': means['response_balanced_accuracy_mean'], 'onset_MAE': means['e2e_onset_mae_sec_mean'], 'peak_MAE': means['e2e_peak_mae_sec_mean'], 'offset_MAE': means['e2e_offset_mae_sec_mean'], 'duration_MAE': means['e2e_duration_mae_sec_mean'], 'selection_loss': means['e2e_selection_loss_mean']})

def compact_stats(stats: pd.DataFrame) -> pd.DataFrame:
    return stats[['component', 'metric_label', 'full_mean', 'neutralized_mean', 'mean_gain_positive_full_better', 'ci95_low', 'ci95_high', 'participants_full_better', 'participants_equal', 'participants_neutralized_better', 'raw_signflip_p', 'holm_p_within_component_6', 'holm_p_global_24']]

def merge_functional_necessity_results():
    mount_drive()
    COMBINED_OUTPUT.mkdir(parents=True, exist_ok=True)
    print('=' * 180)
    print('HEMOBOUNDARY — FUNCTIONAL NECESSITY 10-SEED MERGE + PRINT')
    print('=' * 180)
    print('Model fitting                  : NO')
    print('Nested selection               : NO')
    print('Manual path input              : NO')
    print('Expected seeds                 : 10')
    print('Expected participants/seed     : 26')
    print('Expected variants              : 5')
    print('Statistical unit         : participant after averaging 10 seeds')
    print('Bootstrap reps                 :', BOOT_REPS)
    print('Sign-flip reps                 :', SIGNFLIP_REPS)
    print('Output                         :', COMBINED_OUTPUT)
    print()
    metrics, audit = load_metrics()
    save_csv(audit, COMBINED_OUTPUT / 'COMPLETION_AUDIT.csv')
    validate_metrics(metrics, audit)
    print('COMPLETION CHECK               : PASS')
    print('Participant metric rows        :', len(metrics), '/ 1300')
    metrics = metrics.sort_values(['repeat_seed', 'participant', 'variant']).reset_index(drop=True)
    save_csv(metrics, COMBINED_OUTPUT / 'MERGED_PARTICIPANT_METRICS_ALL_10_SEEDS.csv')
    print('\nMerging per-trial predictions...')
    trials = load_trials()
    trials = trials.sort_values(['repeat_seed', 'participant', 'variant', 'trial_index']).reset_index(drop=True)
    save_csv(trials, COMBINED_OUTPUT / 'MERGED_PER_TRIAL_PREDICTIONS_ALL_10_SEEDS.csv.gz')
    print('Per-trial rows                 :', len(trials), '/ 46800')
    avg = participant_average(metrics)
    avg = avg.sort_values(['participant', 'variant']).reset_index(drop=True)
    save_csv(avg, COMBINED_OUTPUT / 'PARTICIPANT_AVERAGED_METRICS.csv')
    means = means_table(avg)
    stats = paired_table(avg)
    seedwise = seedwise_table(metrics)
    save_csv(means, COMBINED_OUTPUT / 'TABLE1_FUNCTIONAL_NECESSITY_MEANS.csv')
    save_csv(stats, COMBINED_OUTPUT / 'TABLE2_FUNCTIONAL_NECESSITY_PAIRED_STATS.csv')
    save_csv(seedwise, COMBINED_OUTPUT / 'TABLE3_SEEDWISE_MEANS.csv')
    pmeans = compact_means(means)
    pstats = compact_stats(stats)
    print('\n' + '=' * 180)
    print('TABLE 1 — Participant-level means AFTER 10-SEED AVERAGING WITHIN PARTICIPANT')
    print('=' * 180)
    print(pmeans.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    print('\nGain convention: POSITIVE = intact FULL HemoBoundary is better.')
    print('BA gain = FULL - neutralized.')
    print('Timing/loss gain = neutralized - FULL.')
    print('\n' + '=' * 180)
    print('TABLE 2 — PARTICIPANT-LEVEL PAIRED FUNCTIONAL NECESSITY TESTS')
    print('=' * 180)
    print(pstats.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    print('\n' + '=' * 180)
    print('TABLE 3 — SEEDWISE ROBUSTNESS MEANS')
    print('=' * 180)
    print(seedwise.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    focus_metric = {'A — Temporal Boundary Estimation': 'e2e_onset_mae_sec', 'B — Recovery Estimation': 'e2e_offset_mae_sec', 'C — Response Detection': 'response_balanced_accuracy', 'LBR — Local Boundary Refinement': 'e2e_selection_loss'}
    focus_parts = []
    for component, metric in focus_metric.items():
        q = stats[(stats['component'] == component) & (stats['metric'] == metric)]
        if len(q) == 1:
            focus_parts.append(q)
    focus = pd.concat(focus_parts, ignore_index=True) if focus_parts else pd.DataFrame()
    print('\n' + '=' * 180)
    print('COMPONENT-ROLE FOCUS TABLE')
    print('=' * 180)
    if len(focus):
        print(focus[['component', 'metric_label', 'mean_gain_positive_full_better', 'ci95_low', 'ci95_high', 'raw_signflip_p', 'holm_p_within_component_6', 'holm_p_global_24']].to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    summary = ['HEMOBOUNDARY — FUNCTIONAL NECESSITY — 10-SEED MERGE', '=' * 100, '', 'COMPLETION', 'Seeds: 10/10', 'Participants per seed: 26/26', 'Variants: 5/5', f'Participant metric rows: {len(metrics)}/1300', f'Per-trial rows: {len(trials)}/46800', '', 'PRIMARY MEANS', pmeans.to_string(index=False, float_format=lambda x: f'{x:.6f}'), '', 'PAIRED PARTICIPANT-LEVEL TESTS', pstats.to_string(index=False, float_format=lambda x: f'{x:.6f}'), '', 'GAIN CONVENTION', 'Positive gain always means intact FULL HemoBoundary performed better.', 'BA = FULL - neutralized.', 'Timing MAE / selection loss = neutralized - FULL.', '', 'INFERENCE', f'Bootstrap repetitions: {BOOT_REPS}', f'Monte-Carlo sign-flip repetitions: {SIGNFLIP_REPS}', 'Statistical unit: participant after averaging all 10 seeds within participant.', 'Holm: within each component across 6 primary metrics and globally across 24 tests.']
    (COMBINED_OUTPUT / 'SUMMARY.txt').write_text('\n'.join(summary), encoding='utf-8')
    print('\n' + '=' * 180)
    print('Combined results: PASS')
    print('=' * 180)
    print('Saved:', COMBINED_OUTPUT)


In [ ]:
# Cell 5: Run the functional-necessity workflow
run_all_functional_necessity_seeds()
merge_functional_necessity_results()
